In [ ]:
# Cell 1: Setup, Models, and DB Connection
import os
import time
import logging

# 1. Lock to GPU 3 as requested
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"] = "2,3"

import re
import json
import torch
import chromadb
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from huggingface_hub import login # 🟢 NEW IMPORT

# 🟢 AUTHENTICATE WITH HUGGING FACE
login("YOUR_HF_TOKEN") # <--- Paste your actual token here

# --- SETUP LOGGING ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)-8s | %(message)s',
    handlers=[
        logging.FileHandler("phi4_execution.log", encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

if torch.cuda.is_available():
    logger.info(f"✅ GPU Locked To: {torch.cuda.get_device_name(0)} (Physical GPU 3)")
else:
    logger.error("❌ No GPU detected.")

# Connect to ChromaDB
logger.info("🔗 Connecting to ChromaDB...")
client = chromadb.PersistentClient(path="poetry_db")
collection = client.get_collection(name="hindwi_poems") 

# Load Embedder
logger.info("⏳ Loading Embedder...")
embedder = SentenceTransformer('intfloat/multilingual-e5-large', device='cuda')

# Load PHI-4-Hindi (15B Parameters)
logger.info("⏳ Loading 1024m/PHI-4-Hindi...")
MODEL_ID = "1024m/PHI-4-Hindi"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, 
    torch_dtype=torch.bfloat16, 
    device_map="cuda", 
    low_cpu_mem_usage=True
)
model.eval()
logger.info("✅ Phi-4-Hindi loaded successfully!")

/home/literature/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-25 05:14:49,720 | INFO     | ✅ GPU Locked To: NVIDIA RTX A6000 (Physical GPU 3)
2026-02-25 05:14:49,721 | INFO     | 🔗 Connecting to ChromaDB...
2026-02-25 05:14:49,738 | INFO     | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-25 05:14:49,925 | INFO     | ⏳ Loading Embedder...
2026-02-25 05:14:49,928 | INFO     | Load pretrained SentenceTransformer: intfloat/multilingual-e5-large
2026-02-25 05:14:55,886 | INFO     | ⏳ Loading 1024m/PHI-4-Hindi...
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|██████████| 6/6 [00:05<00:00,  1.06it/s]
2026-02-25 05:15:04,070 | INFO     | ✅ Phi-4-Hindi loaded successfully!


In [2]:
# Cell 2: PoetryExperimenter Class
class PoetryExperimenter:
    def __init__(self, model, tokenizer, collection, embedder, logger):
        self.model = model
        self.tokenizer = tokenizer
        self.collection = collection
        self.embedder = embedder
        self.logger = logger

    def _generate(self, system_prompt, user_prompt, temperature=0.7): 
        start_time = time.time()
        
        # Ensure padding token is set
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
            
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
        
        # Standard chat template (No enable_thinking needed)
        inputs = self.tokenizer.apply_chat_template(
            messages, 
            tokenize=True, 
            add_generation_prompt=True, 
            return_tensors="pt",
            return_dict=True
        ).to(self.model.device)
        
        input_length = inputs['input_ids'].shape[1] 
        safe_max_new_tokens = 4096 - input_length - 10 # Phi-4 uses a 4k context window
        
        if safe_max_new_tokens < 500:
            safe_max_new_tokens = 1024 
            
        self.logger.info(f"   [Inference] Prompt: {input_length} tokens | Allowed Gen: {safe_max_new_tokens} tokens")
            
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs, 
                max_new_tokens=safe_max_new_tokens, 
                temperature=temperature, 
                top_p=0.9,
                do_sample=True, 
                eos_token_id=self.tokenizer.eos_token_id,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        clean_output = self.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True).strip()
        
        elapsed_time = time.time() - start_time
        self.logger.info(f"   [Inference] Completed in {elapsed_time:.2f} seconds.")
        
        return clean_output

    def find_best_poet(self, topic):
        try:
            query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
            results = self.collection.query(query_embeddings=query_vector, n_results=1)
            if results['metadatas'] and results['metadatas'][0]:
                poet = results['metadatas'][0][0]['poet_slug']
                self.logger.info(f"   [RAG] Found best poet match: {poet}")
                return poet
        except Exception as e:
            self.logger.error(f"⚠️ Error finding best poet: {e}")
        return "ramdhari-singh-dinkar" 

    def zero_shot(self, topic): 
        return self._generate("आप एक उत्कृष्ट हिंदी कवि हैं।", f"विषय: '{topic}' पर एक कविता लिखें।")

    def few_shot(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        examples_text = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                examples_text += f"उदाहरण {i+1}:\n{text[:200]}...\n\n"
        sys = "आप एक प्रख्यात हिंदी कवि हैं। आपका काम दी गई शैली को समझना और उसी अंदाज़ में एक नई रचना करना है।"
        user = f"यहाँ कुछ उदाहरण दिए गए हैं:\n{examples_text}\nअब, '{topic}' विषय पर एक नई और मौलिक (original) कविता लिखें।"
        return self._generate(sys, user)

    def rag_style_conditioned(self, topic, style="ramdhari-singh-dinkar"):
        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
        res = self.collection.query(query_embeddings=query_vector, n_results=2, where={"poet_slug": style})
        context = ""
        if res['documents'] and res['documents'][0]:
            for i, text in enumerate(res['documents'][0]):
                context += f"संदर्भ {i+1}:\n{text[:300]}...\n"
        user = f"इस शैली का गहराई से अध्ययन करें:\n{context}\n\nअब '{topic}' विषय पर अपनी कल्पना से उसी शैली में एक बिल्कुल नई कविता लिखें।"
        return self._generate("आप एक रचनात्मक और मौलिक (original) कवि हैं।", user)
    
    def plan_then_generate(self, topic):
        sys = "आप एक कवि और आलोचक हैं। पहले कविता की योजना बनाएं, फिर कविता लिखें।"
        user = f"विषय: '{topic}'\n1. भाव (Mood) तय करें。\n2. 5 मुख्य शब्द चुनें。\n3. अलंकार सोचें。\n4. अंत में 'कविता:' शीर्षक के साथ कविता लिखें।"
        return self._generate(sys, user)

    def self_critique(self, topic):
        draft = self.zero_shot(topic)
        critique = self._generate("आप एक कठोर आलोचक हैं।", f"इस कविता की आलोचना करें और 2 कमियां निकालें:\n{draft}")
        final = self._generate("आप एक मास्टर कवि हैं जो अपनी गलतियों को सुधारता है।", f"मूल कविता:\n{draft}\n\nआलोचना:\n{critique}\n\nआलोचना को ध्यान में रखते हुए एक श्रेष्ठ संस्करण लिखें।")
        return f"--- DRAFT ---\n{draft}\n\n--- CRITIQUE ---\n{critique}\n\n--- FINAL ---\n{final}"

    def constraint_based(self, topic):
        user = f"विषय: '{topic}'\nनियम:\n1. ठीक 4 पंक्तियां (lines) होनी चाहिए。\n2. 'आसमान' शब्द का प्रयोग वर्जित है。\n3. अंतिम पंक्ति 'कहानी' शब्द पर खत्म होनी चाहिए।"
        return self._generate("आप नियमों का सख्ती से पालन करने वाले कवि हैं।", user)

    def temp_experiment(self, topic):
        low_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=0.2)
        high_temp = self._generate("आप एक कवि हैं।", f"विषय: '{topic}'", temperature=1.1)
        return f"--- TEMP 0.2 (Predictable) ---\n{low_temp}\n\n--- TEMP 1.1 (Creative) ---\n{high_temp}"

    def persona_based(self, topic):
        return self._generate("आप 19वीं सदी के एक उदास, दार्शनिक कवि हैं जो पुरानी हिंदी में लिखते हैं।", f"इस विषय पर अपने विचार प्रकट करें: '{topic}'")

    def prompt_variants(self, topic):
        variant_a = self._generate("कवि बनो।", f"{topic} पर लिखो।")
        variant_b = self._generate("आप साहित्य अकादमी पुरस्कार विजेता हैं। आपकी भाषा मर्मस्पर्शी है।", f"कृपया '{topic}' विषय पर एक मर्मस्पर्शी रचना प्रस्तुत करें।")
        return f"--- BASIC PROMPT ---\n{variant_a}\n\n--- ENGINEERED PROMPT ---\n{variant_b}"

    def multi_agent(self, topic):
        stanza_1 = self._generate("आप 'कवि A' हैं। आप बहुत ही शांत हैं। विषय पर केवल पहली 4 पंक्तियां लिखें।", f"विषय: '{topic}'")
        stanza_2 = self._generate("आप 'कवि B' हैं। आपका स्वभाव उग्र है। 'कवि A' की कविता को आगे बढ़ाते हुए अगली 4 पंक्तियां लिखें।", f"कवि A ने यह लिखा है:\n{stanza_1}\n\nअब आप इसे अपने विद्रोही अंदाज में पूरा करें।")
        return f"--- STANZ 1 (Calm Agent) ---\n{stanza_1}\n\n--- STANZA 2 (Fiery Agent) ---\n{stanza_2}"

    def auto_eval(self, topic, generated_poem, poet_name, reference_poems, num_evals=5):
        import json
        import ollama
        import statistics

        self.logger.info("   [Evaluation] Starting Dual-LLM evaluation via Ollama...")

        system_instruction = (
            "You are an expert Hindi literary critic and NLP evaluation judge. "
            "Your task is to evaluate a generated Hindi poem. "
            "You must output ONLY a valid JSON object. Do not include markdown formatting, explanations, or introductory text."
        )
        
        evaluation_prompt = (
            f"Topic: {topic}\n"
            f"Target Poet Style: {poet_name}\n\n"
            f"--- REFERENCE POEMS BY {poet_name} ---\n"
            f"{reference_poems}\n\n"
            f"--- GENERATED POEM TO EVALUATE ---\n"
            f"{generated_poem}\n\n"
            "Evaluate the generated poem on a scale of 1 to 10 for the following metrics:\n"
            "1. 'fluency': Language fluency, correct Hindi grammar, and natural rhythm.\n"
            "2. 'coherence': Logical flow, structural integrity, and how well the stanzas connect.\n"
            "3. 'relevance': How accurately it addresses the exact Topic.\n"
            "4. 'creativity': Originality of metaphors, vivid imagery, and avoiding cliches.\n"
            "5. 'style_similarity': How closely the vocabulary, tone, and sentence structure match the Reference Poems provided above.\n\n"
            "Return EXACTLY this JSON format and nothing else:\n"
            '{"fluency": 0, "coherence": 0, "relevance": 0, "creativity": 0, "style_similarity": 0}'
        )

        final_scores = {"Llama-3.1": {}, "Gemma-2": {}}
        metrics = ['fluency', 'coherence', 'relevance', 'creativity', 'style_similarity']

        def clean_json(text):
            text = text.strip()
            if text.startswith("```json"): text = text[7:]
            if text.endswith("```"): text = text[:-3]
            match = re.search(r'\{.*\}', text, re.DOTALL)
            return match.group(0) if match else text

        for model_name, dict_key in [('llama3.1', 'Llama-3.1'), ('gemma2', 'Gemma-2')]:
            raw_scores = {m: [] for m in metrics}
            
            for i in range(num_evals):
                try:
                    response = ollama.chat(model=model_name, messages=[
                        {'role': 'system', 'content': system_instruction},
                        {'role': 'user', 'content': evaluation_prompt}
                    ], options={'temperature': 0.7})
                    
                    clean_text = clean_json(response['message']['content'])
                    parsed = json.loads(clean_text)
                    
                    for m in metrics:
                        if m in parsed:
                            raw_scores[m].append(float(parsed[m]))
                except Exception as e:
                    pass 
            
            aggregated = {}
            for m in metrics:
                vals = raw_scores[m]
                if len(vals) > 1:
                    aggregated[m] = {
                        "mean": round(statistics.mean(vals), 2),
                        "std_dev": round(statistics.stdev(vals), 2)
                    }
                elif len(vals) == 1:
                    aggregated[m] = {"mean": round(vals[0], 2), "std_dev": 0.0}
                else:
                    aggregated[m] = {"error": "Failed all 5 attempts"}
            
            final_scores[dict_key] = aggregated

        return json.dumps(final_scores, indent=2, ensure_ascii=False)

    def run_all(self, topics):
        experiments = {
            "Zero-Shot": self.zero_shot,
            "Few-Shot": self.few_shot,
            "RAG Style (Best Match)": self.rag_style_conditioned,
            "Plan-Then-Generate": self.plan_then_generate,
            "Self-Critique": self.self_critique,
            "Constraints": self.constraint_based,
            "Temperature (0.2 vs 1.1)": self.temp_experiment,
            "Persona (19th Century)": self.persona_based,
            "Prompt Variants": self.prompt_variants,
            "Multi-Agent": self.multi_agent
        }

        output_dir = "Phi4_Output"
        os.makedirs(output_dir, exist_ok=True)
        self.logger.info(f"📂 Execution started. Saving to '{output_dir}/'")
        
        for topic in topics:
            self.logger.info(f"\n{'='*50}\n🌟 STARTING TOPIC: {topic}\n{'='*50}")
            
            best_poet_slug = self.find_best_poet(topic)
            safe_filename = re.sub(r'[^\w\s-]', '', topic).strip().replace(' ', '_')
            file_path = os.path.join(output_dir, f"{safe_filename}.txt")
            
            with open(file_path, "w", encoding="utf-8") as f:
                f.write(f"🌟 TOPIC: {topic}\n")
                f.write(f"🎯 BEST POET MATCH: '{best_poet_slug}'\n")
                f.write(f"{'='*50}\n\n")
                
                for exp_name, exp_func in experiments.items():
                    self.logger.info(f"🧪 Running Experiment: {exp_name}")
                    f.write(f"🧪 EXPERIMENT: {exp_name}\n")
                    f.write(f"{'-'*50}\n")
                    
                    try:
                        query_vector = self.embedder.encode([f"कविता: {topic}"], convert_to_tensor=False)
                        res = self.collection.query(query_embeddings=query_vector, n_results=3, where={"poet_slug": best_poet_slug})
                        
                        reference_context = ""
                        if res['documents'] and res['documents'][0]:
                            for i, text in enumerate(res['documents'][0]):
                                reference_context += f"Reference {i+1}:\n{text[:300]}...\n"

                        if exp_name in ["RAG Style (Best Match)", "Few-Shot"]:
                            poem = exp_func(topic, style=best_poet_slug)
                        else:
                            poem = exp_func(topic)
                        
                        eval_scores_json = self.auto_eval(
                            topic=topic, 
                            generated_poem=poem, 
                            poet_name=best_poet_slug, 
                            reference_poems=reference_context
                        )
                        
                        f.write("📜 GENERATED POEM:\n")
                        f.write(poem + "\n\n")
                        f.write("🧠 DUAL-AI EVALUATION (JSON):\n")
                        f.write(eval_scores_json + "\n\n")
                        f.write(f"{'='*50}\n\n")
                        
                    except Exception as e:
                        self.logger.error(f"❌ FAILED! Error in {exp_name}: {e}")
                        f.write(f"❌ ERROR GENERATING POEM: {str(e)}\n\n")
                        f.write(f"{'='*50}\n\n")
            
            self.logger.info(f"💾 Completed topic. Saved to: {file_path}")

        self.logger.info("✅ All experiments complete!")

In [ ]:
# Cell 3: Run the Massive Experiment Suite
TEST_TOPICS = [
    "🌿 प्रकृति (Nature): बारिश की पहली बूंद",
    "❤️ भावनात्मक (Emotional): अधूरी मोहब्बत",
    "🌍 सामाजिक (Social): नारी शक्ति",
    "🧠 दार्शनिक (Philosophical): समय का चक्र",
    "🎭 रचनात्मक / अनोखा (Creative): टूटी हुई घड़ी की कहानी",
    "🌅 आशावादी (Optimistic): ख्वाबों का आसमान",
    "😔 उदास (Sad): सूनी राहें",
    "🌾 नॉस्टैल्जिक (Nostalgic): मिट्टी की खुशबू",
    "💔 दर्दभरा (Painful): चुप्पी का बोझ",
    "✨ प्रेरणादायक (Inspirational): उम्मीद की किरण",
    "🏡 स्मृतिपूर्ण (Reminiscent): बचपन की गलियाँ",
    "🌆 अकेलापन (Lonely): अजनबी शहर",
    "❤️ रोमांटिक (Romantic): दिल की दस्तक",
    "🤝 भावुक (Emotional): रिश्तों की डोर",
    "⏳ दार्शनिक (Reflective): वक्त की रेत",
    "🕊️ उत्साहपूर्ण (Energetic): सपनों की उड़ान",
    "🎈 निराशाजनक (Hopeless): टूटी हुई पतंग",
    "🪞 गंभीर (Serious): सच का आईना",
    "📜 विरहपूर्ण (Separation): आख़िरी ख़त",
    "🌄 सकारात्मक (Positive): नई सुबह"
]

# Initialize and Run
experimenter = PoetryExperimenter(model, tokenizer, collection, embedder, logger)
experimenter.run_all(TEST_TOPICS)

2026-02-25 05:15:04,119 | INFO     | 📂 Execution started. Saving to 'Phi4_Output/'
2026-02-25 05:15:04,123 | INFO     | 
🌟 STARTING TOPIC: 🌿 प्रकृति (Nature): बारिश की पहली बूंद
Batches: 100%|██████████| 1/1 [00:00<00:00,  2.81it/s]
2026-02-25 05:15:05,224 | INFO     |    [RAG] Found best poet match: gopalkrishna-kaul
2026-02-25 05:15:05,225 | INFO     | 🧪 Running Experiment: Zero-Shot
Batches: 100%|██████████| 1/1 [00:00<00:00, 128.84it/s]
2026-02-25 05:15:05,311 | INFO     |    [Inference] Prompt: 104 tokens | Allowed Gen: 3982 tokens
2026-02-25 05:15:47,659 | INFO     |    [Inference] Completed in 42.36 seconds.
2026-02-25 05:15:47,740 | INFO     |    [Evaluation] Starting Dual-LLM evaluation via Ollama...
2026-02-25 05:16:02,103 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-25 05:16:10,604 | INFO     | HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
2026-02-25 05:16:16,170 | INFO     | HTTP Request: POST http://127.0.

: 